In [1]:
import os
import zipfile
import gdown

# Step 1: Download the dataset zip file from Google Drive
# You must use the direct file ID in the link
drive_file_id = "1PpvvncnQkgDNeBMKVgG2zFYuRhbL873g"
url = f"https://drive.google.com/uc?id={drive_file_id}"
output_path = "/kaggle/working/ijcai2019.zip"

# Download using gdown
gdown.download(url, output_path, quiet=False)

# Step 2: Unzip the file
unzip_dir = "/kaggle/working/ijcai2019_data"
os.makedirs(unzip_dir, exist_ok=True)

with zipfile.ZipFile(output_path, 'r') as zip_ref:
    zip_ref.extractall(unzip_dir)

print(f"Unzipped dataset to: {unzip_dir}")


Downloading...
From (original): https://drive.google.com/uc?id=1PpvvncnQkgDNeBMKVgG2zFYuRhbL873g
From (redirected): https://drive.google.com/uc?id=1PpvvncnQkgDNeBMKVgG2zFYuRhbL873g&confirm=t&uuid=3489d3cc-1bf0-44b9-af7c-54990afb63db
To: /kaggle/working/ijcai2019.zip
100%|██████████| 1.03G/1.03G [00:08<00:00, 122MB/s]


Unzipped dataset to: /kaggle/working/ijcai2019_data


In [2]:
import os
import json
import time
import requests
import re
import zipfile

def load_text_only_dataset(txt_file_path):
    with open(txt_file_path, "r", encoding="utf-8") as f:
        lines = f.readlines()
    assert len(lines) % 4 == 0, "File format incorrect: each sample must have 4 lines"
    samples = []
    for i in range(0, len(lines), 4):
        raw_sentence = lines[i].strip()
        aspect = lines[i + 1].strip()
        sentiment = int(lines[i + 2].strip())
        if "$T$" in raw_sentence:
            sentence = raw_sentence.replace("$T$", aspect)
        else:
            sentence = raw_sentence
        samples.append({
            "raw_sentence": raw_sentence,
            "sentence": sentence,
            "aspect_term": aspect,
            "sentiment": sentiment
        })
    return samples

def clean_aspect(aspect):
    aspect = aspect.strip()
    aspect = re.sub(r"^[#@]+\s*", "", aspect)
    aspect = aspect.lower()
    return aspect

# Paths to your datasets
paths = {
    "train": "/kaggle/working/ijcai2019_data/IJCAI2019_data/twitter2015/train.txt",
    "dev": "/kaggle/working/ijcai2019_data/IJCAI2019_data/twitter2015/dev.txt",
    "test": "/kaggle/working/ijcai2019_data/IJCAI2019_data/twitter2015/test.txt",
}

# Load all samples and clean aspects
all_samples = []
for split, path in paths.items():
    print(f"Loading {split} samples...")
    samples = load_text_only_dataset(path)
    for s in samples:
        s["aspect_term"] = clean_aspect(s["aspect_term"])
    all_samples.extend(samples)

print(f"Total samples loaded: {len(all_samples)}")

# Extract unique cleaned aspects
unique_aspects = set(s["aspect_term"] for s in all_samples)
print(f"Unique cleaned aspects found: {len(unique_aspects)}")

# Function to fetch ConceptNet neighbors (online)
def get_conceptnet_neighbors(word, k=20):
    url = f"http://api.conceptnet.io/related/c/en/{word}"
    params = {'filter': '/c/en'}
    try:
        data = requests.get(url, params=params).json()
        return [edge['@id'].split('/')[-1].replace('_', ' ') for edge in data.get('related', [])[:k]]
    except Exception as e:
        print(f"Error fetching neighbors for '{word}': {e}")
        return []

# Build ConceptNet cache for all cleaned aspects
conceptnet_cache = {}
for aspect in unique_aspects:
    print(f"Fetching ConceptNet neighbors for: {aspect}")
    neighbors = get_conceptnet_neighbors(aspect, k=20)
    conceptnet_cache[aspect] = neighbors
    time.sleep(0.2)  # be polite to API

# === STEP 1: Save to JSON in /kaggle/working/conceptnet_data/ ===
json_dir = "/kaggle/working/conceptnet_data"
os.makedirs(json_dir, exist_ok=True)
json_path = os.path.join(json_dir, "conceptnet_cache_all_splits_cleaned.json")

with open(json_path, "w") as f:
    json.dump(conceptnet_cache, f, indent=2)

print(f"✅ JSON saved to: {json_path}")

# === STEP 2: Zip the JSON file and save in /kaggle/working/zipped_output/ ===
zip_dir = "/kaggle/working/zipped_output"
os.makedirs(zip_dir, exist_ok=True)
zip_path = os.path.join(zip_dir, "conceptnet_cache.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(json_path, arcname="conceptnet_cache_all_splits_cleaned.json")

print(f"✅ Zipped JSON saved to: {zip_path}")


Loading train samples...
Loading dev samples...
Loading test samples...
Total samples loaded: 5338
Unique cleaned aspects found: 3983
Fetching ConceptNet neighbors for: aloha
Fetching ConceptNet neighbors for: christine allen
Fetching ConceptNet neighbors for: martti ahtisaari
Fetching ConceptNet neighbors for: koutoubia garden
Fetching ConceptNet neighbors for: mohamud sadaq
Fetching ConceptNet neighbors for: kirsten
Fetching ConceptNet neighbors for: europe
Fetching ConceptNet neighbors for: gandhi
Fetching ConceptNet neighbors for: iowa state
Fetching ConceptNet neighbors for: wikipedia
Fetching ConceptNet neighbors for: friday the 13th
Fetching ConceptNet neighbors for: sammy
Fetching ConceptNet neighbors for: harlem
Fetching ConceptNet neighbors for: brt
Fetching ConceptNet neighbors for: islanders
Fetching ConceptNet neighbors for: scaleio
Fetching ConceptNet neighbors for: millions for trayvon
Fetching ConceptNet neighbors for: rahul gandhi
Fetching ConceptNet neighbors for: kap